In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

# ==========================================
# Database Connection
# ==========================================

conn = sqlite3.connect("database.db")
cursor = conn.cursor()

# ==========================================
# Create Tables
# ==========================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    skills TEXT,
    salary REAL,
    experience INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS candidates (
    candidate_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    job_role TEXT,
    skills TEXT,
    match_percentage REAL,
    status TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    date TEXT,
    status TEXT
)
""")

conn.commit()

# ==========================================
# Add Employee
# ==========================================

def add_employee():

    try:

        employee_id = int(input("Enter Employee ID: "))
        name = input("Enter Name: ")
        department = input("Enter Department: ")
        skills = input("Enter Skills: ")
        salary = float(input("Enter Salary: "))
        experience = int(input("Enter Experience (Years): "))

        # Duplicate check
        cursor.execute(
            "SELECT * FROM employees WHERE employee_id=?",
            (employee_id,)
        )

        if cursor.fetchone():
            print("Employee ID already exists.")
            return

        cursor.execute("""
        INSERT INTO employees
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            employee_id,
            name,
            department,
            skills,
            salary,
            experience
        ))

        conn.commit()

        print("Employee added successfully.")

    except ValueError:
        print("Invalid input.")


# ==========================================
# View Employees
# ==========================================

def view_employees():

    cursor.execute("SELECT * FROM employees")

    employees = cursor.fetchall()

    print("\n========== EMPLOYEE LIST ==========")

    for emp in employees:

        print(f"""
Employee ID : {emp[0]}
Name        : {emp[1]}
Department  : {emp[2]}
Skills      : {emp[3]}
Salary      : {emp[4]}
Experience  : {emp[5]} years
----------------------------------------
""")


# ==========================================
# Update Employee
# ==========================================

def update_employee():

    try:

        employee_id = int(input("Enter Employee ID to update: "))

        cursor.execute(
            "SELECT * FROM employees WHERE employee_id=?",
            (employee_id,)
        )

        if not cursor.fetchone():
            print("Employee not found.")
            return

        new_department = input("Enter New Department: ")
        new_salary = float(input("Enter New Salary: "))

        cursor.execute("""
        UPDATE employees
        SET department=?, salary=?
        WHERE employee_id=?
        """, (
            new_department,
            new_salary,
            employee_id
        ))

        conn.commit()

        print("Employee updated successfully.")

    except ValueError:
        print("Invalid input.")


# ==========================================
# Delete Employee
# ==========================================

def delete_employee():

    try:

        employee_id = int(input("Enter Employee ID to delete: "))

        cursor.execute(
            "DELETE FROM employees WHERE employee_id=?",
            (employee_id,)
        )

        conn.commit()

        print("Employee deleted successfully.")

    except ValueError:
        print("Invalid input.")


# ==========================================
# Search Employee
# ==========================================

def search_employee():

    name = input("Enter Employee Name: ")

    cursor.execute("""
    SELECT * FROM employees
    WHERE name LIKE ?
    """, ('%' + name + '%',))

    employees = cursor.fetchall()

    if employees:

        for emp in employees:

            print(f"""
Employee ID : {emp[0]}
Name        : {emp[1]}
Department  : {emp[2]}
Skills      : {emp[3]}
Salary      : {emp[4]}
Experience  : {emp[5]} years
""")

    else:
        print("Employee not found.")


# ==========================================
# Candidate Registration
# ==========================================

def register_candidate():

    name = input("Enter Candidate Name: ")
    job_role = input("Enter Job Role: ")
    skills = input("Enter Skills: ")

    required_skills = {
        "python developer": ["python", "sql", "api"],
        "frontend developer": ["react", "javascript", "css"],
        "data analyst": ["python", "excel", "sql"]
    }

    role = job_role.lower()

    if role not in required_skills:
        print("Job role not available.")
        return

    candidate_skills = [
        skill.strip().lower()
        for skill in skills.split(",")
    ]

    required = required_skills[role]

    matched = 0

    for skill in required:

        if skill in candidate_skills:
            matched += 1

    match_percentage = (matched / len(required)) * 100

    status = "Selected" if match_percentage >= 60 else "Rejected"

    cursor.execute("""
    INSERT INTO candidates (
        name,
        job_role,
        skills,
        match_percentage,
        status
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        name,
        job_role,
        skills,
        match_percentage,
        status
    ))

    conn.commit()

    print(f"""
Candidate Registered Successfully.

Match Percentage : {match_percentage}%
Status           : {status}
""")


# ==========================================
# View Candidates
# ==========================================

def view_candidates():

    cursor.execute("""
    SELECT * FROM candidates
    ORDER BY match_percentage DESC
    """)

    candidates = cursor.fetchall()

    print("\n========== CANDIDATE LIST ==========")

    for c in candidates:

        print(f"""
Candidate ID    : {c[0]}
Name            : {c[1]}
Job Role        : {c[2]}
Skills          : {c[3]}
Match Percentage: {c[4]}%
Status          : {c[5]}
----------------------------------------
""")


# ==========================================
# AI Interview Question Generator
# ==========================================

def generate_questions():

    role = input("Enter Job Role: ").lower()

    questions = {
        "python developer": [
            "What is Python?",
            "Explain REST API.",
            "What is SQL JOIN?"
        ],

        "frontend developer": [
            "What is React?",
            "Explain Virtual DOM.",
            "Difference between HTML and HTML5?"
        ],

        "data analyst": [
            "What is data cleaning?",
            "Explain pandas library.",
            "Difference between CSV and Excel?"
        ]
    }

    if role in questions:

        print("\nInterview Questions:")

        for q in questions[role]:
            print("-", q)

    else:
        print("Role not found.")


# ==========================================
# Attendance System
# ==========================================

def mark_attendance():

    try:

        employee_id = int(input("Enter Employee ID: "))
        status = input("Present/Absent: ")

        date = datetime.now().strftime("%Y-%m-%d")

        cursor.execute("""
        INSERT INTO attendance (
            employee_id,
            date,
            status
        )
        VALUES (?, ?, ?)
        """, (
            employee_id,
            date,
            status
        ))

        conn.commit()

        print("Attendance marked successfully.")

    except ValueError:
        print("Invalid input.")


# ==========================================
# Attendance Report
# ==========================================

def attendance_report():

    cursor.execute("""
    SELECT * FROM attendance
    """)

    records = cursor.fetchall()

    print("\n========== ATTENDANCE REPORT ==========")

    for r in records:

        print(f"""
Employee ID : {r[1]}
Date        : {r[2]}
Status      : {r[3]}
""")


# ==========================================
# Export Reports
# ==========================================

def export_reports():

    employee_df = pd.read_sql_query(
        "SELECT * FROM employees",
        conn
    )

    employee_df.to_csv(
        "employee_report.csv",
        index=False
    )

    candidate_df = pd.read_sql_query(
        "SELECT * FROM candidates",
        conn
    )

    candidate_df.to_csv(
        "shortlisted_candidates.csv",
        index=False
    )

    print("Reports exported successfully.")


# ==========================================
# Main Menu
# ==========================================

def main():

    while True:

        print("""
========================================
 SMART HR & AI INTERVIEW MANAGEMENT
========================================

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit
""")

        choice = input("Enter Choice: ")

        if choice == "1":
            add_employee()

        elif choice == "2":
            view_employees()

        elif choice == "3":
            update_employee()

        elif choice == "4":
            delete_employee()

        elif choice == "5":
            search_employee()

        elif choice == "6":
            register_candidate()

        elif choice == "7":
            view_candidates()

        elif choice == "8":
            generate_questions()

        elif choice == "9":
            mark_attendance()

        elif choice == "10":
            attendance_report()

        elif choice == "11":
            export_reports()

        elif choice == "12":

            print("Exiting System...")
            break

        else:
            print("Invalid choice.")


# ==========================================
# Run Program
# ==========================================

if __name__ == "__main__":
    main()


 SMART HR & AI INTERVIEW MANAGEMENT

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit



Enter Choice:  1
Enter Employee ID:  101
Enter Name:  Rahul
Enter Department:  IT
Enter Skills:  Python,SQL
Enter Salary:  50000
Enter Experience (Years):  2


Employee added successfully.

 SMART HR & AI INTERVIEW MANAGEMENT

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit



Enter Choice:  2



========== EMPLOYEE LIST ==========

Employee ID : 1
Name        : Rosan
Department  : CSE
Skills      : Developer
Salary      : 50000.0
Experience  : 1 years
----------------------------------------


Employee ID : 2
Name        : Subit
Department  : IT
Skills      : Tester
Salary      : 60000.0
Experience  : 2 years
----------------------------------------


Employee ID : 101
Name        : Rahul
Department  : IT
Skills      : Python,SQL
Salary      : 50000.0
Experience  : 2 years
----------------------------------------


 SMART HR & AI INTERVIEW MANAGEMENT

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit



...


========== ATTENDANCE REPORT ==========

Employee ID : 101
Date        : 2026-05-28
Status      : Present


 SMART HR & AI INTERVIEW MANAGEMENT

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit



Enter Choice:  11


Reports exported successfully.

 SMART HR & AI INTERVIEW MANAGEMENT

1. Add Employee
2. View Employees
3. Update Employee
4. Delete Employee
5. Search Employee
6. Register Candidate
7. View Candidates
8. AI Interview Questions
9. Mark Attendance
10. Attendance Report
11. Export Reports
12. Exit



Enter Choice:  12


Exiting System...
